# Online BoTorch Optimization Campaign

This notebook runs the new simulator-agnostic `OptimizerBoTorch` path. The optimizer consumes `BCLFOptimizationSpec` directly, trains BoTorch on normalized actual simulator geometry, and records both requested and actual geometry.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from optimization.candidate_space import ContinuousCandidateSpace
from optimization.optimization_spec import BCLFOptimizationSpec
from optimization.optimizer_bo import OptimizerBoTorch
from optimization.recorder import ExperimentRecorder
from optimization.simulators import ServerQueueSimulator
from simulation.result_store import SimulationResultStore

print('repo root:', ROOT)


## Configuration

Set `RUN_ONLINE = True` after checking queue paths and template settings. The notebook is safe to open without launching HFSS/server jobs.

In [ ]:
RUN_ONLINE = False

OUTPUT_DIR = ROOT / 'experiments' / 'bo_online_botorch'
QUEUE_ROOT = ROOT / 'server_queue'
RESULT_STORE_DIR = ROOT / 'experiments' / 'simulation_cache'

VARIABLE_ORDER = ['L', 'L1', 'L2', 'S1', 'S2', 'W', 'W1', 'W2']
VARIABLE_BOUNDS = {
    'L': (7.0, 8.5),
    'L1': (7.0, 8.5),
    'L2': (7.0, 8.5),
    'S1': (0.10, 0.80),
    'S2': (0.10, 0.80),
    'W': (0.50, 1.20),
    'W1': (0.50, 1.20),
    'W2': (0.50, 1.20),
}
UNIT_MAP = {name: 'mm' for name in VARIABLE_ORDER}

SEEDS = [0]
BUDGET = 20
N_INIT = 5
POOL_SIZE = 256
MM_RESOLUTION = 0.01


In [ ]:
def make_live_plot():
    fig, ax = plt.subplots(figsize=(8, 4))
    def callback(trial, history):
        df = pd.DataFrame(history)
        ax.clear()
        ax.plot(df['iteration'], df['objective'], marker='o', label='objective')
        ax.plot(df['iteration'], df['best_so_far'], marker='s', label='best_so_far')
        ax.set_xlabel('iteration')
        ax.set_ylabel('objective (minimize)')
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.canvas.draw_idle()
    return callback


In [ ]:
run_results = []

if RUN_ONLINE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for seed in SEEDS:
        run_id = f'bo_online_botorch_seed{seed}'
        candidate_space = ContinuousCandidateSpace(
            variable_order=VARIABLE_ORDER,
            variable_bounds=VARIABLE_BOUNDS,
            pool_size=POOL_SIZE,
            seed=seed,
        )
        simulator = ServerQueueSimulator(
            queue_root=str(QUEUE_ROOT),
            variable_order=VARIABLE_ORDER,
            unit_map=UNIT_MAP,
            template_name='bclf',
            port_num=2,
            result_store=SimulationResultStore(RESULT_STORE_DIR),
            run_id=run_id,
            mm_resolution=MM_RESOLUTION,
        )
        optimizer = OptimizerBoTorch(
            candidate_space=candidate_space,
            simulator=simulator,
            optimization_spec=BCLFOptimizationSpec(),
            recorder=ExperimentRecorder(output_dir=OUTPUT_DIR),
            budget=BUDGET,
            n_init=N_INIT,
            seed=seed,
            run_id=run_id,
            checkpoint_path=OUTPUT_DIR / f'{run_id}.checkpoint.json',
            progress_callback=make_live_plot(),
        )
        result = optimizer.run()
        run_results.append(result)
        print(run_id, 'best=', result.state.best_objective, 'json=', result.recorder_output.json_path)
else:
    print('RUN_ONLINE is False; set it to True after confirming queue/template settings.')


In [ ]:
if run_results:
    rows = []
    for result in run_results:
        for trial in result.history:
            row = dict(trial)
            row['run_id'] = result.run_id
            rows.append(row)
    df = pd.DataFrame(rows)
    display(df[['run_id', 'iteration', 'phase', 'objective', 'best_so_far', 'requested_geometry', 'geometry', 'geometry_normalized']].tail(20))

    fig, ax = plt.subplots(figsize=(9, 4))
    for run_id, group in df.groupby('run_id'):
        ax.plot(group['iteration'], group['best_so_far'], marker='o', label=run_id)
    ax.set_title('Online BoTorch best-so-far')
    ax.set_xlabel('iteration')
    ax.set_ylabel('objective (minimize)')
    ax.grid(True, alpha=0.3)
    ax.legend()
